<a href="https://colab.research.google.com/github/jefferyocran/FraudGuard-OXGBoost/blob/main/baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U "shap>=0.46" "xgboost>=2.1" -q

In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
import shap

SEED = 42
np.random.seed(SEED)
print("Environment ready. Seed =", SEED)


Environment ready. Seed = 42


In [4]:
# --- Load the public UCI dataset ---
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
uci = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])
uci['label'] = uci['label'].map({'ham': 0, 'spam': 1})
uci['source'] = 'uci_public'

# --- Load your Ghanaian field data ---
# (upload ghana_momo_sms.csv to Colab first, then this reads it)
field = pd.read_csv('ghana_momo_sms.csv')

# --- Combine them ---
df = pd.concat([uci, field], ignore_index=True)

print("UCI messages:      ", len(uci))
print("Field messages:    ", len(field))
print("Combined total:    ", len(df))
print("\nLabel balance:")
print(df['label'].value_counts())
print("\nField data by label:")
print(field['label'].value_counts())


UCI messages:       5572
Field messages:     30
Combined total:     5602

Label balance:
label
0    4830
1     772
Name: count, dtype: int64

Field data by label:
label
1    25
0     5
Name: count, dtype: int64


In [5]:
# Separate the messages (X) from the labels (y)
X_text = df['message']
y = df['label']

# Convert text to TF-IDF number features
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2))
X = vectorizer.fit_transform(X_text)

print("Shape of feature matrix:", X.shape)

Shape of feature matrix: (5602, 1000)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print("Training messages:", X_train.shape[0])
print("Testing messages:", X_test.shape[0])

Training messages: 4481
Testing messages: 1121


In [7]:
# Model 1: Standard XGBoost (the main control)
xgb_model = xgb.XGBClassifier(
    max_depth=6, n_estimators=200, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

# Model 2: Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=SEED
)
rf_model.fit(X_train, y_train)

# Model 3: Logistic Regression
lr_model = LogisticRegression(
    C=1.0, class_weight='balanced', max_iter=1000, random_state=SEED
)
lr_model.fit(X_train, y_train)

print("All three baseline models trained.")

All three baseline models trained.


In [8]:
def evaluate(model, X_test, y_test, name):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    print(f"\n{name}")
    print("  Accuracy: ", round(accuracy_score(y_test, preds), 4))
    print("  Macro F1: ", round(f1_score(y_test, preds, average='macro'), 4))
    print("  AUC-ROC:  ", round(roc_auc_score(y_test, probs), 4))
    print("  AUC-PR:   ", round(average_precision_score(y_test, probs), 4))

evaluate(xgb_model, X_test, y_test, "Standard XGBoost")
evaluate(rf_model, X_test, y_test, "Random Forest")
evaluate(lr_model, X_test, y_test, "Logistic Regression")


Standard XGBoost
  Accuracy:  0.9759
  Macro F1:  0.9454
  AUC-ROC:   0.9864
  AUC-PR:    0.9597

Random Forest
  Accuracy:  0.975
  Macro F1:  0.9429
  AUC-ROC:   0.9895
  AUC-PR:    0.9668

Logistic Regression
  Accuracy:  0.9777
  Macro F1:  0.9526
  AUC-ROC:   0.9901
  AUC-PR:    0.9734


In [9]:
import pickle

# Save the models and the exact data splits so O-XGBoost uses identical data
with open('baseline_artifacts.pkl', 'wb') as f:
    pickle.dump({
        'vectorizer': vectorizer,
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'xgb_model': xgb_model, 'rf_model': rf_model, 'lr_model': lr_model
    }, f)

print("Baseline locked and saved. Do not modify this notebook again.")


Baseline locked and saved. Do not modify this notebook again.


In [10]:
def check_message(text, model):
    # Convert the new message using the SAME vectorizer from training
    features = vectorizer.transform([text])
    pred = model.predict(features)[0]
    prob = model.predict_proba(features)[0][1]  # probability of scam
    label = "SCAM" if pred == 1 else "LEGITIMATE"
    print(f"Message: {text}")
    print(f"Prediction: {label}  (scam probability: {round(prob*100, 1)}%)\n")

# Try some messages the model has never seen
check_message("Your MoMo wallet is blocked. Send your PIN now to unlock it.", xgb_model)
check_message("Hey, are we still meeting at 3pm today?", xgb_model)
check_message("Congratulations! You won GH¢5000. Pay GH¢50 to claim your prize.", xgb_model)


Message: Your MoMo wallet is blocked. Send your PIN now to unlock it.
Prediction: LEGITIMATE  (scam probability: 14.899999618530273%)

Message: Hey, are we still meeting at 3pm today?
Prediction: LEGITIMATE  (scam probability: 0.20000000298023224%)

Message: Congratulations! You won GH¢5000. Pay GH¢50 to claim your prize.
Prediction: LEGITIMATE  (scam probability: 47.0%)

